### Note:
- Use GPT-5 model for summarizing articles
- Because the budget is limited, we test with 200 first articles.

In [1]:
import json
import os
import time
from datasets import load_dataset
from openai import OpenAI
from tqdm import tqdm  # Thư viện tạo thanh tiến trình (pip install tqdm)

# 1. Cấu hình Client (Apifree)
client = OpenAI(
    base_url="https://api.apifree.ai/v1",
    api_key="sk-p7AP3tEpX3GkNTrp0KMedJNXwYZ1R"
)
MODEL_ID = "openai/gpt-5"

# 2. Load dataset
print(">> Đang tải dataset...")
# Load tập train, không cần load hết nếu mạng chậm (nhưng thư viện này thường load cached)
ds = load_dataset("hihihohohehe/vifactcheck-normalized", split="train")

# Tên file lưu kết quả
OUTPUT_FILE = "summaries_train_result.json"

>> Đang tải dataset...


In [ ]:
def summarize(input_text):
    prompt = f"""
    Tóm tắt văn bản sau thành MỘT câu duy nhất (tối đa 30 từ).
    Yêu cầu: Giữ số liệu/keyword, bao phủ nội dung chính, không giải thích thừa.

    Văn bản:
    <<<
    {input_text}
    >>>
    """

    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.6,
            max_tokens=10000, # Giảm xuống vì chỉ cần tóm tắt ngắn (tiết kiệm token)
            stream=False
        )
        content = response.choices[0].message.content
        
        # Logic cũ của bạn trả về finish_reason là chưa đúng, ta cần trả về content
        if content:
            return content.strip()
        else:
            return "Error: Empty response"

    except Exception as e:
        return f"Error: {str(e)}"

# 3. Xử lý Main Loop
def main():
    # A. Load dữ liệu cũ nếu đã chạy trước đó (Cơ chế Resume)
    results = {}
    if os.path.exists(OUTPUT_FILE):
        print(f">> Tìm thấy file cũ '{OUTPUT_FILE}'. Đang tải dữ liệu đã làm...")
        try:
            with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
                results = json.load(f)
        except json.JSONDecodeError:
            print(">> File cũ bị lỗi, sẽ tạo mới.")
            results = {}
    
    print(f">> Đã hoàn thành: {len(results)}/{len(ds)} dòng.")

    # B. Duyệt qua dataset
    # Sử dụng tqdm để hiển thị thanh tiến trình
    for i, item in tqdm(enumerate(ds), total=len(ds), desc="Processing"):
        index_str = str(i)

        # Nếu index này đã có trong file json rồi thì bỏ qua (không gọi API nữa -> tiết kiệm tiền)
        if index_str in results:
            continue

        input_text = item.get('Context', "")

        # Gọi hàm tóm tắt
        summary_text = summarize(input_text)
        
        # Lưu vào dict bộ nhớ tạm
        results[index_str] = summary_text

        # C. Lưu ngay lập tức vào file JSON
        # Việc này giúp bạn mở file lên xem realtime và không mất data nếu crash
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        
        # (Tuỳ chọn) Sleep nhẹ để tránh bị Rate Limit nếu gọi quá nhanh
        time.sleep(0.5) 

    print(f">> Hoàn tất! Kết quả lưu tại {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

>> Tìm thấy file cũ 'summaries_train_result.json'. Đang tải dữ liệu đã làm...
>> Đã hoàn thành: 184/5062 dòng.


Processing:   7%|▋         | 346/5062 [18:41<1:03:15,  1.24it/s] 